In [1]:
!pip install -q flask requests

In [24]:
import os
import joblib
import threading
import time
import requests
import pandas as pd

from flask import Flask, request, jsonify

print("Libraries imported successfully!")

Libraries imported successfully!


In [13]:
model_path = "/content/faculty_burnout_model_pipeline.pkl"

if os.path.exists(model_path):
    print("Model file found successfully!")
else:
    print("Model file NOT found!")

Model file found successfully!


In [14]:
model = joblib.load(model_path)

print("Trained model loaded successfully!")
print("Model type:", type(model))

Trained model loaded successfully!
Model type: <class 'sklearn.pipeline.Pipeline'>


In [26]:
from flask import Flask, request, jsonify

app = Flask(__name__)

print("Flask application created successfully!")

Flask application created successfully!


In [27]:
@app.route("/", methods=["GET"])
def home():
    return jsonify({
        "message": "Faculty Burnout & Workload Balancer API is running",
        "status": "success"
    })

In [28]:
@app.route("/predict", methods=["POST"])
def predict_api():

    try:
        data = request.get_json()

        if not data:
            return jsonify({
                "error": "Request body must contain JSON data"
            }), 400

        required_fields = [
            "Teaching_Hours",
            "Advising_Students",
            "Committee_Count",
            "Research_Hours",
            "Admin_Hours",
            "Semester_Progress",
            "Historical_Leave_Days"
        ]

        for field in required_fields:
            if field not in data:
                return jsonify({
                    "error": f"Missing field: {field}"
                }), 400

        input_data = pd.DataFrame([{
            "Teaching_Hours": float(data["Teaching_Hours"]),
            "Advising_Students": int(data["Advising_Students"]),
            "Committee_Count": int(data["Committee_Count"]),
            "Research_Hours": float(data["Research_Hours"]),
            "Admin_Hours": float(data["Admin_Hours"]),
            "Semester_Progress": float(data["Semester_Progress"]),
            "Historical_Leave_Days": float(data["Historical_Leave_Days"])
        }])

        prediction = model.predict(input_data)[0]

        probabilities = model.predict_proba(input_data)[0]
        class_names = model.classes_

        probability_result = {
            str(class_name): round(float(probability), 4)
            for class_name, probability in zip(class_names, probabilities)
        }

        return jsonify({
            "predicted_burnout_risk": str(prediction),
            "risk_probabilities": probability_result
        })

    except Exception as e:
        return jsonify({
            "error": str(e)
        }), 400

In [31]:
def run_flask():
    app.run(
        host="0.0.0.0",
        port=8001,
        debug=False,
        use_reloader=False
    )

flask_thread = threading.Thread(
    target=run_flask,
    daemon=True
)

flask_thread.start()

time.sleep(3)

print("Flask server started successfully!")
print("Server running on port 8001")

 * Serving Flask app '__main__'
 * Debug mode: off


INFO:werkzeug:WARNING: This is a development server. Do not use it in a production deployment. Use a production WSGI server instead.
 * Running on all addresses (0.0.0.0)
 * Running on http://127.0.0.1:8001
 * Running on http://172.28.0.12:8001
INFO:werkzeug:Press CTRL+C to quit


Flask server started successfully!
Server running on port 8001


In [32]:
response = requests.get("http://127.0.0.1:8001/")

print("Status Code:", response.status_code)
print("Response:", response.json())

INFO:werkzeug:127.0.0.1 - - [18/Sep/2026 06:20:03] "GET / HTTP/1.1" 200 -


Status Code: 200
Response: {'message': 'Faculty Burnout & Workload Balancer API is running', 'status': 'success'}


In [33]:
test_data = {
    "Teaching_Hours": 20,
    "Advising_Students": 10,
    "Committee_Count": 2,
    "Research_Hours": 10,
    "Admin_Hours": 5,
    "Semester_Progress": 0.7,
    "Historical_Leave_Days": 3
}

response = requests.post(
    "http://127.0.0.1:8001/predict",
    json=test_data
)

print("Status Code:", response.status_code)
print("\nResponse:")
print(response.json())

INFO:werkzeug:127.0.0.1 - - [18/Sep/2026 06:20:52] "POST /predict HTTP/1.1" 200 -


Status Code: 200

Response:
{'predicted_burnout_risk': 'Low', 'risk_probabilities': {'High': 0.0, 'Low': 0.5351, 'Medium': 0.4649}}


In [34]:
high_workload_data = {
    "Teaching_Hours": 35,
    "Advising_Students": 25,
    "Committee_Count": 6,
    "Research_Hours": 5,
    "Admin_Hours": 12,
    "Semester_Progress": 0.9,
    "Historical_Leave_Days": 10
}

response = requests.post(
    "http://127.0.0.1:8001/predict",
    json=high_workload_data
)

print("Status Code:", response.status_code)
print("\nResponse:")
print(response.json())

INFO:werkzeug:127.0.0.1 - - [18/Sep/2026 06:22:16] "POST /predict HTTP/1.1" 200 -


Status Code: 200

Response:
{'predicted_burnout_risk': 'High', 'risk_probabilities': {'High': 0.6028, 'Low': 0.0067, 'Medium': 0.3906}}


In [35]:
invalid_data = {
    "Teaching_Hours": 20,
    "Advising_Students": 10
}

response = requests.post(
    "http://127.0.0.1:8001/predict",
    json=invalid_data
)

print("Status Code:", response.status_code)
print("Response:", response.json())

INFO:werkzeug:127.0.0.1 - - [18/Sep/2026 06:22:55] "POST /predict HTTP/1.1" 400 -


Status Code: 400
Response: {'error': 'Missing field: Committee_Count'}


In [36]:
print(app.url_map)

Map([<Rule '/static/<filename>' (OPTIONS, HEAD, GET) -> static>,
 <Rule '/' (OPTIONS, HEAD, GET) -> home>,
 <Rule '/predict' (POST, OPTIONS) -> predict_api>])


In [37]:
print("===== Faculty Burnout & Workload Balancer API =====")
print()
print("Base URL: http://127.0.0.1:8001")
print()
print("Available Endpoints:")
print("1. GET  /")
print("   - Checks whether the API is running")
print()
print("2. POST /predict")
print("   - Predicts faculty burnout risk")
print("   - Returns risk probabilities")
print()
print("Required Input Fields:")
print("- Teaching_Hours")
print("- Advising_Students")
print("- Committee_Count")
print("- Research_Hours")
print("- Admin_Hours")
print("- Semester_Progress")
print("- Historical_Leave_Days")

===== Faculty Burnout & Workload Balancer API =====

Base URL: http://127.0.0.1:8001

Available Endpoints:
1. GET  /
   - Checks whether the API is running

2. POST /predict
   - Predicts faculty burnout risk
   - Returns risk probabilities

Required Input Fields:
- Teaching_Hours
- Advising_Students
- Committee_Count
- Research_Hours
- Admin_Hours
- Semester_Progress
- Historical_Leave_Days
